# DOE 샘플링 (파이프라인 0~1단계)

`docs/GUIDELINE.md` 파이프라인에서 **제일 먼저 실행하는 노트북**. 설계변수·범위가 정해지면 여기서 DOE 점을 생성해서 `doe_points.csv`로 내보내고, 그걸 카티아(설계변수 열)·아바쿠스(불확실 변수 열) 자동화에 넘긴다. 해석 결과(SEA)까지 다 모으면 그 결과를 `PDD_bumper_SEA.ipynb`, `Kriging_bumper_SEA.ipynb`의 데이터 로딩 자리에 투입.

**주의**: DOE는 설계변수뿐 아니라 불확실 변수(noise, 예: 충돌각 편차·재료 물성 배율)까지 같이 흔들어서 뽑아야 함 — 서로게이트가 나중에 "설계값 고정 + noise만 흔들어 mean/std 계산"을 하려면, 학습 데이터에 noise가 흔들리는 경우들이 포함돼 있어야 함.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import qmc


def lhs_design(domain_min, domain_max, n_samples, seed=0):
    # Latin Hypercube Sampling으로 DOE 점 생성. 카티아/아바쿠스에 넘길 실제 물리 단위로 반환.
    # domain_min, domain_max: (dim, 1) 배열
    dim = domain_min.shape[0]
    sampler = qmc.LatinHypercube(d=dim, seed=seed)
    unit_samples = sampler.random(n=n_samples)  # (n_samples, dim), [0,1) 구간

    lo = domain_min.ravel()
    hi = domain_max.ravel()
    physical = lo + unit_samples * (hi - lo)  # (n_samples, dim)

    return physical.T  # (dim, n_samples) 관례 유지


def calculate_basis_num(N, m, S):
    # PDD 항(계수) 개수를 미리 계산 (공유용 REDLAB_UQ.py의 calculate_basis_num 참고).
    # 필요 샘플 수 계획에 사용: 필요 샘플 수 ~ 이 값의 2~3배.
    import math
    basis_count = 1
    for s in range(1, S + 1):
        basis_count += math.comb(N, s) * math.comb(m, s)
    return basis_count

## 1. 변수 정의 (설계변수 + 불확실 변수)

In [ ]:
# TODO: 실제 변수 이름·순서·범위로 교체할 것. 설계변수를 앞쪽, 불확실 변수를 뒤쪽에 두는 걸 추천
# (PDD_bumper_SEA.ipynb / Kriging_bumper_SEA.ipynb의 DESIGN_IDX/NOISE_IDX와 순서를 맞추기 위함).
#
# 예시 (두께 t, 높이 h = 설계변수 / 충돌각 편차 theta, 재료물성 배율 E = 불확실 변수):
# variable_names = ["t", "h", "theta", "E"]
# domain_min = np.array([[1.5], [80], [-15], [0.98]])
# domain_max = np.array([[3.0], [120], [15], [1.02]])
# n_design = 2  # 앞의 n_design개가 설계변수 (DESIGN_IDX = list(range(n_design)))

variable_names = []  # TODO
domain_min = None    # TODO, (dim, 1)
domain_max = None    # TODO, (dim, 1)
n_design = None       # TODO, 설계변수 개수 (앞쪽 n_design개)

## 2. 샘플 수 결정

경험칙: 변수 수의 8~10배로 시작. PDD를 쓸 계획이면 `calculate_basis_num`으로 항 개수를 먼저 확인하고 그 2~3배로 잡는 것도 방법.

In [ ]:
# 예시: 변수 4개, PDD를 n=3, y=2로 쓸 계획이면
# print(calculate_basis_num(N=4, m=3, S=2))  # 이 값의 2~3배를 n_samples로

n_samples = 60  # TODO: 변수 개수/예산에 맞게 조정

## 3. DOE 생성 및 시각화 (공간이 고르게 채워지는지 확인)

In [ ]:
X_doe = lhs_design(domain_min, domain_max, n_samples, seed=0)  # (dim, n_samples)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(variable_names) - 1, figsize=(4 * (len(variable_names) - 1), 4))
if len(variable_names) == 2:
    axes = [axes]
for i, ax in enumerate(axes):
    ax.scatter(X_doe[0], X_doe[i + 1], s=15)
    ax.set_xlabel(variable_names[0])
    ax.set_ylabel(variable_names[i + 1])
plt.tight_layout()
plt.show()

## 4. CSV로 내보내기 (카티아/아바쿠스 자동화가 참조할 파일)

In [ ]:
doe_df = pd.DataFrame(X_doe.T, columns=variable_names)
doe_df.to_csv("doe_points.csv", index=False)
print(f"{n_samples}개 DOE 점을 doe_points.csv로 저장함 (컬럼: {variable_names})")

# TODO: 카티아 매크로는 앞쪽 n_design개 열(설계변수)로 형상 생성,
#       아바쿠스는 나머지 열(불확실 변수)로 해석 조건 설정.
# TODO: 전체 DOE 점 해석이 끝나면 SEA 결과를 이 표 옆에 Y 열로 붙여
#       doe_results.csv로 저장 -> PDD_bumper_SEA.ipynb / Kriging_bumper_SEA.ipynb에서 로드.